In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import cv2
import yaml
import numpy as np
from glob import glob
from sklearn.model_selection import train_test_split

# Paths
DATASET_PATH = "/kaggle/input/polypdb/PolypDB/PolypDB/PolypDB_center_wise"
OUTPUT_PATH = "/kaggle/working/yolo_dataset"

# Allowed image and mask extensions
image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".tiff")
mask_extensions = (".png", ".jpg", ".jpeg", ".bmp", ".tiff")

# Create necessary directories
for split in ["train", "val", "test"]:
    os.makedirs(f"{OUTPUT_PATH}/images/{split}", exist_ok=True)
    os.makedirs(f"{OUTPUT_PATH}/labels/{split}", exist_ok=True)

# Function to get bounding boxes from masks
def get_bounding_boxes(mask):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = []
    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        x_center, y_center = x + w / 2, y + h / 2
        boxes.append([x_center, y_center, w, h])
    return boxes

# Collect all valid image-mask pairs with labels
all_data = []
class_names = set()

for center in os.listdir(DATASET_PATH):
    center_path = os.path.join(DATASET_PATH, center)

    if not os.path.isdir(center_path):
        continue  # Skip non-directory files

    for modality in os.listdir(center_path):
        modality_path = os.path.join(center_path, modality)

        img_dir = os.path.join(modality_path, "images")
        mask_dir = os.path.join(modality_path, "masks")

        if not os.path.exists(img_dir) or not os.path.exists(mask_dir):
            continue  # Skip if either images or masks folder is missing

        for img_path in glob(f"{img_dir}/*"):
            if not img_path.endswith(image_extensions):
                continue  # Skip non-image files

            img_name = os.path.basename(img_path)
            img_base = os.path.splitext(img_name)[0]  # Filename without extension

            # Find corresponding mask
            mask_path = None
            for ext in mask_extensions:
                candidate_mask = os.path.join(mask_dir, img_base + ext)
                if os.path.exists(candidate_mask):
                    mask_path = candidate_mask
                    break

            if mask_path:
                label = f"{center}_{modality}"
                class_names.add(label)
                all_data.append((img_path, mask_path, label))
            else:
                print(f"⚠️ No mask found for {img_path}")

# Assign unique class IDs
class_names = sorted(class_names)  # Sort for consistency
class_to_id = {name: idx for idx, name in enumerate(class_names)}

# Split dataset (80% Train, 10% Val, 10% Test)
train_data, test_data = train_test_split(all_data, test_size=0.2, random_state=42)
val_data, test_data = train_test_split(test_data, test_size=0.5, random_state=42)

# Function to save images and labels in YOLO format
def save_data(split, data):
    for img_path, mask_path, label in data:
        img_name = os.path.basename(img_path)
        img = cv2.imread(img_path)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        if img is None or mask is None:
            continue  # Skip corrupted files

        bboxes = get_bounding_boxes(mask)
        if not bboxes:
            continue  # Skip images without bounding boxes

        # Save image
        new_img_path = f"{OUTPUT_PATH}/images/{split}/{img_name}"
        cv2.imwrite(new_img_path, img)

        # Save YOLO label file
        label_id = class_to_id[label]
        label_path = f"{OUTPUT_PATH}/labels/{split}/{img_name.replace('.jpg', '.txt')}"
        with open(label_path, "w") as f:
            for box in bboxes:
                x, y, w, h = box
                H, W = img.shape[:2]
                f.write(f"{label_id} {x/W} {y/H} {w/W} {h/H}\n")

# Save train, val, test sets
save_data("train", train_data)
save_data("val", val_data)
save_data("test", test_data)

# Generate `data.yaml` file dynamically
data_yaml = {
    "train": f"{OUTPUT_PATH}/images/train",
    "val": f"{OUTPUT_PATH}/images/val",
    "test": f"{OUTPUT_PATH}/images/test",
    "nc": len(class_names),
    "names": class_names
}

with open(f"{OUTPUT_PATH}/data.yaml", "w") as yaml_file:
    yaml.dump(data_yaml, yaml_file, default_flow_style=False)

print(f"✅ Dataset converted to YOLO format! Found {len(all_data)} valid image-mask pairs.")
print(f"📂 Classes: {class_names}")


✅ Dataset converted to YOLO format! Found 3934 valid image-mask pairs.
📂 Classes: ['BKAI_BLI', 'BKAI_FICE', 'BKAI_LCI', 'BKAI_WLI', 'Karolinska_WLI', 'Simula_NBI', 'Simula_WLI']


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

# Train the model
model.train(data=f"{OUTPUT_PATH}/data.yaml", epochs=50, imgsz=640)

Ultralytics 8.3.91 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: task=detect, mode=train, model=yolov8s.pt, data=/kaggle/working/yolo_dataset/data.yaml, epochs=50, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train4, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, s

train: Scanning /kaggle/working/yolo_dataset/labels/train... 3144 images, 0 backgrounds, 0 corrupt: 100%|██████████| 3144/3144 [00:03<00:00, 930.35it/s]


train: New cache created: /kaggle/working/yolo_dataset/labels/train.cache
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Scanning /kaggle/working/yolo_dataset/labels/val... 393 images, 0 backgrounds, 0 corrupt: 100%|██████████| 393/393 [00:00<00:00, 771.46it/s]


val: New cache created: /kaggle/working/yolo_dataset/labels/val.cache
Plotting labels to runs/detect/train4/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.000909, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
TensorBoard: model graph visualization added ✅
Image sizes 640 train, 640 val
Using 4 dataloader workers
Logging results to runs/detect/train4
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 13/13 [00:07<00:00,  1.70it/s]


                   all        393      26315      0.864      0.111      0.137     0.0912
              BKAI_BLI          5        443      0.612     0.0113     0.0227     0.0152
             BKAI_FICE          9        959      0.919     0.0104     0.0204     0.0166
              BKAI_LCI          5        341      0.907     0.0147     0.0368     0.0278
              BKAI_WLI        100       9281      0.937     0.0107     0.0265     0.0202
        Karolinska_WLI          6        175          1     0.0274     0.0873     0.0771
            Simula_NBI         18         19      0.753      0.684      0.745      0.466
            Simula_WLI        250      15097      0.919     0.0162     0.0198     0.0156


/usr/local/lib/python3.10/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.10/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


Speed: 0.6ms preprocess, 4.5ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to runs/detect/train4


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7d8432fbb970>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
  

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load training metrics
metrics_path = "/kaggle/working/runs/detect/train4/results.csv"  # Adjust path if needed
df = pd.read_csv(metrics_path)

df.head(50)

,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
0,1,68.961,1.05598,2.47145,1.30652,0.24450,0.05257,0.01718,0.01116,1.08090,1.33396,1.32125,0.000301,0.000301,0.000301
1,2,133.607,1.08451,1.47099,1.31114,0.31559,0.03643,0.02424,0.01603,1.08342,1.33456,1.32882,0.000592,0.000592,0.000592
2,3,198.099,1.19252,1.46122,1.33843,0.78645,0.03973,0.03602,0.01923,1.42160,1.69863,1.34595,0.000872,0.000872,0.000872
3,4,262.315,1.26083,1.43132,1.29877,0.41266,0.03792,0.05617,0.02486,1.35691,1.39708,1.24285,0.000855,0.000855,0.000855
4,5,326.417,1.24157,1.35697,1.28198,0.79480,0.03440,0.03450,0.01956,1.37082,1.39626,1.24533,0.000837,0.000837,0.000837
5,6,390.448,1.20277,1.26905,1.25074,0.42584,0.03557,0.03304,0.02255,1.35240,1.40233,1.22815,0.000819,0.000819,0.000819
6,7,454.414,1.19095,1.25200,1.24099,0.72618,0.05782,0.08414,0.04783,1.25997,1.18085,1.16307,0.000801,0.000801,0.000801
7,8,518.447,1.15043,1.18577,1.21692,0.44294,0.08459,0.06678,0.04401,1.29555,1.14411,1.17172,0.000783,0.000783,0.000783
8,9,582.448,1.15579,1.14971,1.21025,0.60729,0.05103,0.05496,0.03306,1.30615,1.10530,1.17970,0.000765,0.000765,0.000765
9,10,646.507,1.13907,1.10630,1.19797,0.58865,0.07279,0.07834,0.04297,1.31940,1.11036,1.17377,0.000747,0.000747,0.000747


In [ ]:
import shutil

# Zip the saved model directory
model_dir = "/kaggle/working/yolo_dataset"
zip_path = "/kaggle/working/yolodata"
shutil.make_archive(zip_path.replace(".zip", ""), 'zip', model_dir)

# Provide the zip file for download

'/kaggle/working/yolodata.zip'